# Part A: K-means from Scratch

In [ ]:
import numpy as np
import sklearn

In [ ]:
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler

wine  = load_wine()
X_scaled = StandardScaler().fit_transform(wine.data)

def kmeans(X, k, max_iter=100, tol=1e-4, rand_state = 47):
    rng = np.random.default_rng(rand_state)

    #Centroid initialization
    indices = rng.choice(len(X), size=k, replace=False)
    centers = X[indices].copy()

    for iteration in range(max_iter):
        # 2. Sample assignment
        distance = ((X[:,None,:]-centers[None,:,:])**2).sum(axis=2) #计算距离
        labels = np.argmin(distance,axis=1) #找到当前点距离最近的簇中心
        # 3. Centroid update
        new_centers = centers.copy()
        for cluster in range(k):#遍历所有簇
            members = X[labels == cluster]#取出被分到当前簇的所有样本
            if len(members)>0:
                new_centers[cluster] = members.mean(axis=0)#对该簇的样本按列求平均，得到新的聚类中心
        # 4. Stopping condition
        movement = np.linalg.norm(new_centers - centers, axis=1).max()#计算移动最远的中心，只有当移动最远的中心也停止，才算稳定
        centers = new_centers
        if movement<tol:
            break
    distances = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)#用最终的中心重新计算一次样本所属的簇
    labels = np.argmin(distances, axis=1)
    return centers, labels, iteration + 1


In [ ]:
#compare with scikit-learn K-means.
from sklearn.cluster import KMeans
for random in range(47,52):
    for k in [2, 3, 4, 5]:
        model = KMeans(
            n_clusters=k,
            random_state=random
        )
        labels = model.fit_predict(X_scaled)
        print(
            f"K={k}, "
            f"objective value={model.inertia_:.2f}, "
            f"iterations={model.n_iter_}"
        )
    print("------------------------------------------")   


In [ ]:
#k=2，3，4，5，random=47
for k in range(2,6):
    centers, labels, n_iter = kmeans(X_scaled, k=k)
    objective_value = np.sum((X_scaled - centers[labels]) ** 2)
    print(objective_value)
    print(n_iter)

In [ ]:
#k=2，3，4，5，random=48
for k in range(2,6):
    centers, labels, n_iter = kmeans(X_scaled, k=k, rand_state=48)
    objective_value = np.sum((X_scaled - centers[labels]) ** 2)
    print(objective_value)
    print(n_iter)

In [ ]:
##k=2，3，4，5，random=49
for k in range(2,6):
    centers, labels, n_iter = kmeans(X_scaled, k=k,rand_state=49)
    objective_value = np.sum((X_scaled - centers[labels]) ** 2)
    print(objective_value)
    print(n_iter)

In [ ]:
#k=2，3，4，5，random=50
for k in range(2,6):
    centers, labels, n_iter = kmeans(X_scaled, k=k,rand_state=50)
    objective_value = np.sum((X_scaled - centers[labels]) ** 2)
    print(objective_value)
    print(n_iter)

In [ ]:
#k=2，3，4，5，random=51
for k in range(2,6):
    centers, labels, n_iter = kmeans(X_scaled, k=k,rand_state=51)
    objective_value = np.sum((X_scaled - centers[labels]) ** 2)
    print(objective_value)
    print(n_iter)

Why K-means may produce different results with different initializations.

Ans: Different starting centroids can produce different sample assignments. Therefore, different initializations may lead to different final clusters and objective values.

初始中心决定第一次分组，第一次分组决定新的中心，而新的中心又影响下一次分组。 这样一轮轮更新，最初的差异就可能把算法带到不同的最终结果

## Part B: Clustering Evaluation
### Require labels
#### Adjusted Rand Index (ARI)
样本之间的“同组关系”是否正确

ARI = 1：分组完全一致

ARI ≈ 0：接近随机分组的平均水平

ARI < 0：一致性低于随机分组的期望水平

#### Normalized Mutual Information (NMI)
聚类结果包含多少真实类别信息

假如某个簇几乎全是同一种真实类别，那么簇编号提供的信息就很多；如果每个簇都混合了各种真实类别，提供的信息就少。

 1：划分完全一致。

 0：划分之间没有共享信息。

 中间值：反映划分的信息关联程度。

#### Clustering accuracy (ACC)
聚类准确率：聚类完成后，有多少比例的样本分对了

### No require labels
#### Silhouette score
轮廓系数衡量的是：一个样本与自己组里的成员是否靠近，与其他组的成员是否远离。 它不需要真实类别，只根据样本之间的距离评价分组

分数接进1，说明它在自己的组里比较合适。

分数接近0，说明它处在两个组的交界附近，归属不太明确。

分数小于0，更靠近其他组，分组可能不合理。

#### Calinski–Harabasz score
CH衡量的是各组之间是否分得开，同时每组内部是否聚得紧。越大越好。

CH 比较两个量：

 组间离散程度B：各组中心离全体样本的总中心有多远，同时考虑每组的样本数量。越大，说明各组中心越分散。
 
 组内离散程度W：每个样本到自己组中心的距离平方，再全部相加。越小，说明每组内部越紧凑。

#### Davies–Bouldin score
DB衡量的是每组与其他组有多容易混淆。越小越好。

In [ ]:
#Evaluation of the k-means experiment
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score,
)

y_true = wine.target
for rand_state in range(47, 52):
    for k in [2, 3, 4, 5]:
        centers, labels, n_iter = kmeans(
            X_scaled,
            k=k,
            rand_state=rand_state,
        )

        ari = adjusted_rand_score(y_true, labels)
        nmi = normalized_mutual_info_score(y_true, labels)
        silhouette = silhouette_score(X_scaled, labels)

        print(
            f"random_state={rand_state}, K={k}, "
            f"ARI={ari:.4f}, "
            f"NMI={nmi:.4f}, "
            f"Silhouette={silhouette:.4f}"
        )
    print("------------------------------------------")
#K=3时，结果最好

Why is clustering evaluation more difficult than classification evaluation?

Ans: In clustering evaluation, sometimes we don't know the true label so we cannot directly check whether the groups are correct.

Why should ground-truth labels not be used during unsupervised training?

Ans: Because the goal of unsupervised training is to find patterns only used features. Using labels to guide training would make it supervised.

Can an internal metric always identify the most meaningful clustering?

Ans: No. Internal metrics are used to evaluate the seperation and compactness of different groups. But these do not necessarily reflect meaningful real-world categories.

Why is clustering accuracy permutation-invariant?

Ans: Because it uses the best matching between clusters and true classes. Renaming the clusters does not change their members, so it does not change the accuracy.